<b><h1 style="font-size:30px;text-align: center;">- Data Engineering - Exercise 5 -</h1></b>
<b><h1 style="font-size:25px;text-align: center;">- Handling Missing Values -</h1></b>

<div style="text-align:center; margin:16px 0 28px;">
  <img
    src="https://futsal-dinamo.hr/wp-content/uploads/2021/01/Algebra-Bernays.png"
    alt="Algebra Bernays"
    style="width:600px; max-width:60%; height:auto; margin-top:25px;"
  >
</div>

---------------------

<b>*Created: October 2026*</b>

<b>*Author: Mislav Spajić, mag. ing. comp.*</b>

---------------------

# How to work with this notebook

> **First save your own copy:** `File → Save Notebook As…` → `EX_5_my_work.ipynb`, and work in it.
> That way `git pull` never conflicts with your changes.

Every section has the same layout:

| Mark | Meaning |
|---|---|
| 📖 **Theory** | what you need to know for the section – the notebook is self-contained |
| ▶️ **Demo** | code we run together (`Shift + Enter`) |
| 🧩 **Task** | a cell marked `# TODO` – replace `...` with your code |
| ✅ **Check** | run it right after the task; it prints ✅ or explains what is wrong (the checks live in `checks.py`) |
| ✏️ | write your answer in the markdown cell (double-click the cell, then `Shift + Enter`) |
| ⭐ | optional – if you have time left in class, or at home |

Run the cells **in order, top to bottom** – later cells use variables from earlier ones.
Demo cells never depend on your task solutions, so you can skip a task and continue.
If something gets stuck: `Kernel → Restart Kernel and Run All Cells…`, then continue where you were.

The checks never show the expected numbers – they only tell you whether your result is right and give a hint if it is not.

# Contents

| # | Section | Data | ≈ min |
|---|---|---|---|
| 0 | Setup | – | 5 |
| 1 | Missing values you cannot see: sentinels, text codes, structural gaps | cars | 9 |
| 2 | Why is it missing? Testing MCAR, MAR and MNAR with data | Titanic | 14 |
| 3 | Why the mechanism matters: a simulation with known truth | synthetic | 12 |
| 4 | Comparing imputation methods by hiding known values | Titanic | 16 |
| 5 | Missing indicators and data leakage in a pipeline | Titanic | 10 |
| 6 | Time series: fill within each series, and not too far | air quality | 11 |
| 7 | ⭐ Final task: Bronze → Silver cleaning as code | cars | 3 + home |

**Datasets** (folder `data/`):
* `titanic/train.csv` – Titanic passengers ([Kaggle: Titanic](https://www.kaggle.com/c/titanic))
* `air-quality-data-in-india/city_day.csv` – daily air quality in 26 Indian cities, 2015–2020 ([Kaggle](https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india))
* `cars.csv` – technical data and prices of cars ([Kaggle: Car Features and MSRP](https://www.kaggle.com/datasets/CooperUnion/cardataset))
* section 3 generates its own synthetic data, so we know the true values

# 0. Setup (≈ 5 min)

We work in the Jupyter pod (`environment/anaconda_jupyter.yaml`). Anaconda already has `pandas`, `numpy`, `scipy`, `matplotlib`, `seaborn` and `scikit-learn`.
We only add **missingno**, a small library for plotting missing values. If it cannot be installed (no internet), the notebook still works – the missingno plots are skipped.

In [ ]:
# Only if the instructor announced a fix to the materials – pulls the newest version of the repository.
# (You work in your own copy, so your changes are not overwritten.)
!git pull

In [ ]:
%pip install -q missingno

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import missingno as msno
except ImportError:              # e.g. no internet in the pod – the notebook works without missingno plots
    msno = None
    print("missingno is not available – missingno plots will be skipped.")

pd.set_option("display.max_columns", 30)
print("pandas", pd.__version__, "| numpy", np.__version__)

In [ ]:
def missing_table(df):
    """Number and percentage of missing values per column (only columns that have some)."""
    n = df.isna().sum()
    table = pd.DataFrame({"missing": n, "% rows": (100 * n / len(df)).round(1)})
    return table[table["missing"] > 0].sort_values("missing", ascending=False)

In [ ]:
# Task checks live in checks.py (same folder as the notebook).
# reload makes sure a new version is loaded after git pull, without restarting the kernel.
import importlib
import checks
importlib.reload(checks)

In [ ]:
titanic = pd.read_csv("data/titanic/train.csv")
air = pd.read_csv("data/air-quality-data-in-india/city_day.csv", parse_dates=["Date"])
cars = pd.read_csv("data/cars.csv")

for name, df in [("titanic", titanic), ("air", air), ("cars", cars)]:
    print(f"{name:8} {df.shape[0]:6} rows × {df.shape[1]:2} columns")

# 1. Missing values you cannot see (≈ 9 min)

📖 **Theory** – a missing value is a value that is not stored for a variable. Pandas shows it as `NaN` (numbers), `None` (objects) or `NaT` (dates),
and `isna()` finds it. In real data many missing values do **not** look like `NaN`:

| Form | Example | Does `isna()` see it? |
|---|---|---|
| explicit | `NaN`, `None`, empty CSV cell | yes |
| **sentinel number** | `0`, `-1`, `999`, `9999`, a suspiciously common price | no |
| **text code** | `"UNKNOWN"`, `"N/A"`, `"-"`, `""` | no |
| **structural** ("not applicable") | number of cylinders of an electric car | yes, but it is not *unknown* – there is nothing to know |

Hidden missing values break **completeness** and **accuracy** at the same time: a placeholder that looks like a real value quietly enters averages and models.
Structural gaps must not be "imputed" at all – the right value is either a defined code (`0`, `"none"`) or a separate category.

**How to find them:** profile every column – type, share of missing values, number of distinct values and **how often the most common value appears**.
A real measurement is rarely identical for thousands of rows; a placeholder is.

▶️ **Demo** – the car table.

In [ ]:
cars.info()

In [ ]:
print("Fully identical rows (duplicates):", cars.duplicated().sum())
missing_table(cars)

In [ ]:
# Where are the cylinders missing? Group the missing rows by make, model and fuel type.
cars.loc[cars["Engine Cylinders"].isna()].groupby(["Make", "Model", "Engine Fuel Type"]).size()

Electric cars have no cylinders – that gap is **structural**. Mazda RX-7 and RX-8 have a rotary (Wankel) engine, which has no cylinders in the usual sense either.
Neither is "unknown", so neither should get an average number of cylinders.

🧩 **Task 1** – write a reusable column profiler.

In [ ]:
def profile(df):
    """One row per column of df with: dtype, n_missing, pct_missing, n_unique, top_value, top_share."""
    rows = {}
    for col in df.columns:
        s = df[col]
        counts = s.value_counts(normalize=True)          # shares of non-missing values, most common first
        rows[col] = {
            "dtype": str(s.dtype),
            # TODO number of missing values in the column
            "n_missing": ...,
            # TODO percentage (0–100) of ALL rows that are missing
            "pct_missing": ...,
            # TODO number of distinct non-missing values
            "n_unique": ...,
            # TODO the most common value and its share among non-missing values (use `counts`)
            "top_value": ...,
            "top_share": ...,
        }
    return pd.DataFrame(rows).T


prof = profile(cars)
prof.sort_values("n_unique", ascending=False)       # continuous columns first

In [ ]:
# ✅ Check
checks.task_1(profile, cars)

The most common price (`MSRP`) is a suspiciously round number. Is it a real price?
From now on we work on the table without duplicates, `cars_dd`.

🧩 **Task 1 b)**

In [ ]:
cars_dd = cars.drop_duplicates()
cars_dd["MSRP"].value_counts().head(5)

In [ ]:
# TODO 1) (oldest, newest) Year among cars with MSRP == 2000 – a tuple of two numbers
year_range_2000 = ...

# TODO 2) Among cars from year 2000 or older: what share (0–1) has MSRP == 2000?
share_2000_old = ...

# TODO 3) How many cars have the text "UNKNOWN" in "Transmission Type"?
n_unknown_transmission = ...

print(year_range_2000, share_2000_old, n_unknown_transmission)

In [ ]:
# ✅ Check
checks.task_1b(year_range_2000, share_2000_old, n_unknown_transmission)

✏️ **Task 1 c)** For each case: what does the gap or odd value really mean, is it *unknown* or *not applicable*, and what would you do with it?

| Case | What it means | Unknown or not applicable? | What would you do? |
|---|---|---|---|
| `MSRP = 2000` | | | |
| `Transmission Type = "UNKNOWN"` | | | |
| `Engine Cylinders` missing, electric car | | | |
| `Engine Cylinders` missing, Mazda RX-7/RX-8 | | | |
| `Market Category` missing (≈ 30 % of rows) | | | |

# 2. Why is it missing? Testing MCAR, MAR and MNAR with data (≈ 14 min)

📖 **Theory** – let **M** be the *missingness indicator* (`True` where the value is missing). The mechanism describes what M depends on:

| Mechanism | Formally | Meaning | Example |
|---|---|---|---|
| **MCAR** – missing completely at random | P(M \| data) = P(M) | missingness has nothing to do with any variable | questionnaires lost in the mail |
| **MAR** – missing at random | P(M \| data) = P(M \| *observed* data) | missingness is explained by **other, observed** variables | younger people skip the spending question more often – and we know their age |
| **MNAR** – missing not at random | P(M) depends on the **missing value itself** | | people with high income don't report their income |

**What can data tell us?**
* Under MCAR, M is independent of **every** observed variable. So if the missing rate differs between groups of another variable, it is **not MCAR**.
  * categorical variable → compare missing rates by group, **chi-square test of independence** on the table `variable × M`,
  * numeric variable → compare its mean where the value is missing vs. present, **Welch t-test**,
  * all at once → train a classifier that **predicts M** from the other columns; ROC AUC ≈ 0.5 means "unpredictable" (consistent with MCAR), clearly above 0.5 means "not MCAR".
* **MAR vs. MNAR cannot be decided from the data alone**: MNAR depends on values we never saw. Only domain knowledge, the data collection process,
  or extra data (a follow-up survey, another source) can support or reject it. Data can *reject* MCAR; it can never *prove* MAR.

**Read tests with care:**
* A p-value answers "could this difference be chance?", not "how big is it?". Always look at the **rates** too (effect size).
* With **small samples** a test has almost no power: a large p-value does **not** show MCAR, it shows that you cannot tell.
* Tiny groups (a few rows) produce extreme rates by chance; many tests at once produce some "significant" results by chance.

> **Common mistake:** `msno.heatmap` shows whether *missingness of one column* goes together with *missingness of another*.
> It says nothing about whether missingness depends on the **values** of other columns – so it cannot tell you that something is MCAR.

▶️ **Demo** – is `Age` missing equally often in every passenger class (`Pclass`)?

In [ ]:
from scipy.stats import chi2_contingency

age_missing = titanic["Age"].isna()
rates = age_missing.groupby(titanic["Pclass"]).mean()
table = pd.crosstab(titanic["Pclass"], age_missing)          # rows: class, columns: missing False/True
chi2, p_value, dof, expected = chi2_contingency(table)

print((rates * 100).round(1).astype(str) + " %")
print(f"chi-square p-value: {p_value:.1e}")
rates.plot(kind="bar", title="Share of passengers without Age, by class", ylabel="share", rot=0)
plt.show()

In [ ]:
# missingno matrix sorted by class: white lines = missing values
if msno:
    msno.matrix(titanic.sort_values("Pclass"), figsize=(10, 4), fontsize=10)
    plt.show()

In 3rd class `Age` is missing almost 5 times as often as in 2nd class, and the p-value is tiny: missingness depends on `Pclass`, so it is **not MCAR**.

🧩 **Task 2 a)** – turn the demo into a function, so you can test any column against any grouping variable.

In [ ]:
def missingness_test(df, col, by):
    """Share of missing `col` in each group of `by`, and the chi-square p-value of independence."""
    missing = df[col].isna()
    # TODO share of missing values in each group of `by` (a Series indexed by the groups)
    rates = ...
    # TODO p-value of chi2_contingency on the contingency table pd.crosstab(df[by], missing)
    p_value = ...
    return rates, p_value


rates, p = missingness_test(titanic, "Cabin", "Pclass")
print(rates.round(3), "\np-value:", p)

In [ ]:
# ✅ Check – if your function is correct, it also prints the evidence for Age against several variables
checks.task_2a(missingness_test, titanic)

🧩 **Task 2 b)** – `Fare` is numeric, so instead of groups we compare its mean between passengers **with** and **without** `Age`.

In [ ]:
from scipy.stats import ttest_ind

# TODO 1) mean Fare grouped by whether Age is missing (a Series with index False/True)
fare_by_age_missing = ...

# TODO 2) p-value of a Welch t-test (equal_var=False) comparing Fare of the two groups
fare_p = ...

print(fare_by_age_missing.round(2), "\np-value:", fare_p)

In [ ]:
# ✅ Check
checks.task_2b(fare_by_age_missing, fare_p)

⭐ **Task 2 c)** – all variables at once: can a model **predict** where `Age` is missing?
If missingness were MCAR, the best possible ROC AUC would be about 0.5.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

X_m = pd.get_dummies(titanic[["Pclass", "Sex", "SibSp", "Parch", "Fare", "Embarked"]].astype({"Pclass": str}),
                     drop_first=True).astype(float)

# TODO mean ROC AUC of LogisticRegression(max_iter=1000) predicting age_missing from X_m,
#      5-fold StratifiedKFold(5, shuffle=True, random_state=0), scoring="roc_auc"
auc_missing = ...
auc_missing

In [ ]:
# ✅ Check
checks.task_2c(auc_missing)

✏️ **Task 2 d)**

1. Classify each column and name the **evidence** (numbers from the tasks above).

| Column | MCAR / MAR / MNAR | Evidence |
|---|---|---|
| `Age` | | |
| `Cabin` | | |
| `Embarked` (2 missing rows) | | |

2. Could `Age` also be **MNAR**? Describe one realistic story in which the chance of a missing age depends on the age itself,
   and say what *extra data* (not this table) could support or reject it.
3. In the evidence table of task 2 a), the group `SibSp = 8` has 100 % missing ages. Why is that weak evidence?

# 3. Why the mechanism matters: a simulation with known truth (≈ 12 min)

📖 **Theory** – with real data we never know the missing values, so we cannot see how wrong a method is. A **simulation** fixes that:
we generate complete data, delete values with a mechanism we choose, impute, and compare with the truth.

Three single-imputation methods:

| Method | Fills a gap with | Keeps | Loses |
|---|---|---|---|
| **mean imputation** | the mean of the observed values | nothing about relationships | spread (all gaps get one value), correlations |
| **regression imputation** | the prediction of a regression on other variables, ŷ = a + b·x | the relationship with x | spread: every filled point lies exactly **on** the line |
| **stochastic regression imputation** | prediction + random noise with the residual standard deviation, ŷ + ε | relationship **and** spread | individual values are random – fine for distributions, not for "the best guess" for one row |

Regression imputation is the idea behind **MICE / IterativeImputer** (section 4): every column with gaps is regressed on the others, repeatedly.

▶️ **Demo** – 2000 people: income depends on age. We delete about 30 % of the incomes in three different ways.

In [ ]:
rng = np.random.default_rng(2026)
n = 2000
age = rng.uniform(20, 65, n)
income = 1000 + 40 * age + rng.normal(0, 400, n)          # the TRUE incomes – in real life we never see them
sim = pd.DataFrame({"age": age, "income": income})

masks = {                                                   # True = income is missing
    "MCAR": rng.random(n) < 0.3,                                                          # lost in the mail
    "MAR":  rng.random(n) < np.where(age < 35, 0.6, 0.1),                                # the young skip the question
    "MNAR": rng.random(n) < np.where(income > np.quantile(income, 0.7), 0.7, 0.1),      # high earners skip it
}

summary = pd.DataFrame({
    "% missing": {k: 100 * m.mean() for k, m in masks.items()},
    "mean of observed (deletion)": {k: income[~m].mean() for k, m in masks.items()},
}).round(1)
summary["bias vs. true mean"] = (summary["mean of observed (deletion)"] - income.mean()).round(1)
print("true mean:", income.mean().round(1))
summary

Simply dropping the gaps (complete-case analysis) is fine under MCAR, but biased under MAR and MNAR – exactly what the lecture said about deletion.
Can imputation do better?

🧩 **Task 3** – implement (stochastic) regression imputation.

In [ ]:
from sklearn.linear_model import LinearRegression

def regression_impute(x, y, stochastic=False, seed=0):
    """Return a copy of y where NaN values are replaced by a linear regression of y on x."""
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    known = ~np.isnan(y)
    model = LinearRegression()

    # TODO 1) fit the model on the KNOWN rows only (sklearn needs a 2-D x: x[known].reshape(-1, 1))
    ...

    pred = model.predict(x[~known].reshape(-1, 1))
    if stochastic:
        # TODO 2) residual standard deviation on the known rows: std of (y_known - model prediction for x_known)
        resid_sd = ...
        pred = pred + np.random.default_rng(seed).normal(0, resid_sd, size=len(pred))

    y_filled = y.copy()
    y_filled[~known] = pred
    return y_filled


y_obs = np.where(masks["MAR"], np.nan, income)          # incomes we "observed" under MAR
regression_impute(age, y_obs)[:5]

In [ ]:
# ✅ Check – if correct, it prints the comparison of all methods under all three mechanisms
checks.task_3(regression_impute, age, income, masks)

✏️ **Task 3 b)** Use the table printed by the check.

1. Under which mechanism(s) is mean imputation unbiased? Under which does regression imputation remove the bias, and why there?
2. Under MNAR, regression imputation is better than the mean but still biased. Why can no method that uses only the observed data fully fix it?
3. Which method keeps the **spread** (standard deviation) of income? Why does plain regression imputation shrink it?

# 4. Comparing imputation methods by hiding known values (≈ 16 min)

📖 **Theory** – overview of the methods:

<center><img src="images/Missin.png" width="800"/></center>

| Method | Idea | Watch out |
|---|---|---|
| mean / median / mode | one value for every gap | flattens the distribution; ignores other columns |
| **group median** | median within a group (e.g. class, title) | uses a MAR variable; groups need enough rows |
| **KNN imputation** | average of the *k* most similar rows (by distance on the other columns) | **distance needs scaled features** – otherwise the column with the largest numbers (`Fare`) decides everything; with a *single* column there are no "similar rows", so it collapses to the mean |
| **Iterative imputation (MICE)** | regress each column with gaps on the others, repeat in rounds | `IterativeImputer` is still "experimental" in scikit-learn and must be enabled explicitly |

**How to compare without the truth?** Take rows where the value **is known**, hide a random 20 % of them (*masking*), impute, and measure the error
on the hidden values: **RMSE** (punishes big errors) and **MAE** (average absolute error). Repeat with several random masks.
Also compare the **spread** of the imputed values with the true spread – a method can have a low error and still destroy the distribution.

Caveat: hiding values at random simulates MCAR. Real gaps in `Age` are MAR (section 2), so these scores are somewhat optimistic.

▶️ **Demo** – feature table: `Title` (Mr, Mrs, Miss, Master…) is extracted from the passenger name; "Master" was used for boys, so it says a lot about age.

In [ ]:
title = titanic["Name"].str.extract(r",\s*([^.]+)\.", expand=False)
title = title.replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
title = title.where(title.isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")

features = pd.DataFrame({
    "Age": titanic["Age"],
    "Pclass": titanic["Pclass"],
    "SibSp": titanic["SibSp"],
    "Parch": titanic["Parch"],
    "Fare": titanic["Fare"],
    "Sex_male": (titanic["Sex"] == "male").astype(int),
    "Title": title.map({"Mr": 0, "Mrs": 1, "Miss": 2, "Master": 3, "Rare": 4}),
})
known = features[features["Age"].notna()].reset_index(drop=True)     # rows where we know the truth
print(title.value_counts().to_dict())
known.head()

In [ ]:
def mask_known(df, col="Age", frac=0.2, seed=0):
    """Copy of df with a random `frac` of `col` hidden; also returns the positions of the hidden rows."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(df), size=int(frac * len(df)), replace=False)
    masked = df.copy()
    masked.loc[idx, col] = np.nan
    return masked, idx


def evaluate(impute, n_seeds=5):
    """Average RMSE, MAE and spread of an imputation function over several random masks of `known`.
    `impute` gets the masked DataFrame and returns the complete Age column (array or Series)."""
    scores = []
    for seed in range(n_seeds):
        masked, idx = mask_known(known, seed=seed)
        pred = np.asarray(impute(masked), dtype=float)[idx]
        truth = known["Age"].to_numpy()[idx]
        scores.append([np.sqrt(np.mean((pred - truth) ** 2)), np.mean(np.abs(pred - truth)), np.std(pred)])
    return pd.Series(np.mean(scores, axis=0), index=["RMSE", "MAE", "SD of imputed"])


print("SD of the true hidden ages ≈", round(known["Age"].std(), 1))

In [ ]:
from sklearn.impute import KNNImputer

baseline = pd.DataFrame({
    "mean": evaluate(lambda d: d["Age"].fillna(d["Age"].mean())),
    "median": evaluate(lambda d: d["Age"].fillna(d["Age"].median())),
    "median by Pclass": evaluate(lambda d: d["Age"].fillna(d.groupby("Pclass")["Age"].transform("median"))),
    "KNN on Age only": evaluate(lambda d: KNNImputer(n_neighbors=5).fit_transform(d[["Age"]])[:, 0]),
    "KNN, all columns, NOT scaled": evaluate(lambda d: KNNImputer(n_neighbors=5).fit_transform(d)[:, 0]),
}).T.round(2)
baseline

Notice two things:
* **KNN on `Age` alone gives exactly the same result as the mean.** With one column there is nothing to measure similarity on, so every gap gets the average of… everyone.
  (You will find this mistake in many online tutorials.)
* Using all columns helps, but without scaling the distance is dominated by `Fare` (values up to 512) while `Sex_male` (0/1) hardly counts.

🧩 **Task 4** – three better imputers. Each function gets the masked DataFrame and returns the complete `Age` column.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer  # noqa: F401  (enables IterativeImputer)
from sklearn.impute import IterativeImputer


def impute_title_median(df):
    # TODO a) fill Age with the median Age of the same Title (hint: groupby + transform("median"))
    return ...


def impute_knn_scaled(df, k=5):
    scaler = StandardScaler()
    # TODO b) 1) scale all columns (fit_transform; StandardScaler ignores NaN when fitting)
    #         2) KNNImputer(n_neighbors=k) on the scaled data
    #         3) scaler.inverse_transform back to years, return the Age column (position df.columns.get_loc("Age"))
    ...
    return ...


def impute_iterative(df):
    # TODO c) IterativeImputer(random_state=0) on all columns, return the Age column
    ...
    return ...


masked, idx = mask_known(known, seed=0)
print(np.asarray(impute_title_median(masked))[idx][:5])

In [ ]:
# ✅ Check – each part separately; if all are correct it prints the full comparison table
checks.task_4(impute_title_median, impute_knn_scaled, impute_iterative, evaluate, baseline)

✏️ **Task 4 d)**

1. Which method would you put into a production pipeline for `Age`, and why? Weigh error, spread, simplicity and explainability.
2. Why is the gain from the best method over the median smaller than you might expect? (Hint: how much do the other columns really tell us about age?)
3. The masks hid ages **at random**. Real missing ages are concentrated in 3rd class and port Q. How could you make the evaluation more realistic?

# 5. Missing indicators and data leakage in a pipeline (≈ 10 min)

📖 **Theory**
* **Missing indicator:** the fact that a value is missing can itself carry information (an order without a shipping address may be a non-serious buyer).
  `SimpleImputer(add_indicator=True)` adds a 0/1 column `missingindicator_<col>` for every column that had gaps during fitting.
* **Data leakage:** every number used to fill gaps (a mean, a median, a KNN neighbourhood) is *learned from data*. It must be learned from the **training data only**
  and then applied unchanged to validation, test and production data – the future does not exist yet when you train.
* A scikit-learn **`Pipeline`** / **`ColumnTransformer`** does that automatically: in cross-validation, `fit` runs on the training folds only.
  It also makes the cleaning **reproducible**: the same fitted object is saved and used in production.
* The worst leak is **target leakage**: using the label you want to predict (here `Survived`) to fill gaps. At prediction time the label is unknown, so this can never work in production.

▶️ **Demo** – does missing `Age` say something about survival?

In [ ]:
titanic.groupby(titanic["Age"].isna())["Survived"].mean().rename("survival rate").to_frame()

Passengers without a recorded age survived less often – so it is worth keeping the information "Age was missing" as a feature.

🧩 **Task 5 a)** – build the pipeline: numeric columns get median + indicator, categorical columns get the most frequent value, then one-hot encoding and a random forest.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

num_cols = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
cat_cols = ["Sex", "Embarked"]
X, y = titanic[num_cols + cat_cols], titanic["Survived"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

preprocess = ColumnTransformer([
    # TODO numeric: SimpleImputer with the median AND a missing indicator
    ("num", ..., num_cols),
    # TODO categorical: SimpleImputer with the most frequent value, then one-hot encoding
    ("cat", make_pipeline(..., OneHotEncoder(handle_unknown="ignore")), cat_cols),
])
model = make_pipeline(preprocess, RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0))

# TODO mean ROC AUC of `model` with cross_val_score(..., cv=cv, scoring="roc_auc")
auc_pipeline = ...
auc_pipeline

In [ ]:
# ✅ Check
checks.task_5(model, auc_pipeline)

▶️ **Demo** – three ways to fill `Age`, same model, same folds:
1. inside the pipeline (correct),
2. the global median computed on **all** rows **before** cross-validation,
3. a colleague's "improvement": the median age per class **and survival** (`Pclass × Survived`), also before cross-validation.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

X_num = titanic[["Pclass", "Age", "SibSp", "Parch", "Fare"]].assign(Sex_male=(titanic["Sex"] == "male").astype(int))
y_s = titanic["Survived"]
cv_s = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0)

X_before = X_num.assign(Age=X_num["Age"].fillna(X_num["Age"].median()))
X_target = X_num.assign(Age=X_num["Age"].fillna(X_num.groupby([titanic["Pclass"], y_s])["Age"].transform("median")))

leak = pd.Series({
    "1) median inside the pipeline": cross_val_score(make_pipeline(SimpleImputer(strategy="median"), rf), X_num, y_s, cv=cv_s, scoring="roc_auc").mean(),
    "2) global median before CV": cross_val_score(rf, X_before, y_s, cv=cv_s, scoring="roc_auc").mean(),
    "3) median by Pclass × Survived before CV": cross_val_score(rf, X_target, y_s, cv=cv_s, scoring="roc_auc").mean(),
}, name="ROC AUC").round(4)
leak.to_frame()

✏️ **Task 5 b)**

1. Versions 1 and 2 score (almost) the same. Does that mean computing the median before the split is fine? Why is the pipeline still the right design?
2. Version 3 scores clearly higher. Where exactly does the extra performance come from, and what happens to it when the model is used on a new passenger?

# 6. Time series: fill within each series, and not too far (≈ 11 min)

📖 **Theory** – in a time series, neighbouring values are similar, so gaps are filled from neighbours:

| Method | pandas | Idea | Good for |
|---|---|---|---|
| forward fill | `.ffill()` | last known value | values that stay constant until they change (prices, statuses) |
| backward fill | `.bfill()` | next known value | rarely alone – it uses the future |
| linear interpolation | `.interpolate()` | straight line between the two known neighbours | smooth measurements with a trend |
| time interpolation | `.interpolate(method="time")` | like linear, but weighted by real time distance | **irregular** timestamps (identical to linear when every day has a row) |
| polynomial / spline | `.interpolate(method="polynomial", order=2)` | a curve through neighbours | smooth curves; can overshoot |

Rules that matter in practice:
* **Fill within each series.** A table with many sensors / cities / customers stacked below each other must be grouped first – otherwise values leak from one series into the next.
* **Don't fill too far.** `limit=n` fills at most *n* consecutive gaps; `limit_area="inside"` fills only between two known values (no extrapolation at the ends).
  A gap of months is not a gap any more – it is missing data, and inventing it creates fake certainty.
* Strong **seasonality** (daily, weekly, yearly cycles) and long gaps call for seasonal methods (e.g. the value from the same weekday/season), not straight lines.
* `fillna(method="ffill")` is deprecated since pandas 2.1 and **no longer works in pandas 3** – use `.ffill()` / `.bfill()`.

▶️ **Demo** – the methods on a tiny series:

In [ ]:
s = pd.Series([0.0, 1.0, np.nan, np.nan, np.nan, 5.0, 6.0, np.nan])
pd.DataFrame({"original": s, "ffill": s.ffill(), "bfill": s.bfill(),
              "interpolate": s.interpolate(), "interpolate inside, limit=2": s.interpolate(limit=2, limit_area="inside")})

▶️ **Demo** – the air-quality table has **26 cities stacked below each other**, sorted by city and then by date. A frequent mistake: `ffill()` over the whole table.

In [ ]:
cols = air.select_dtypes("number").columns.tolist()          # measurements: PM2.5, PM10, NO, …, AQI
print(air["City"].nunique(), "cities,", air["Date"].min().date(), "–", air["Date"].max().date())

air_naive = air.copy()
air_naive[cols] = air_naive[cols].ffill()

border = air.index[air["City"] == "Aizawl"][0]                # first row of the city Aizawl
print("BEFORE:")
display(air.loc[border - 2: border + 2, ["City", "Date", "PM2.5", "Xylene", "AQI"]])
print("AFTER naive ffill():")
display(air_naive.loc[border - 2: border + 2, ["City", "Date", "PM2.5", "Xylene", "AQI"]])
print("Cells filled by the naive ffill:", (air[cols].isna() & air_naive[cols].notna()).sum().sum())

Aizawl **never measured** Xylene, but the naive `ffill()` gave it Ahmedabad's last value for every single day.

🧩 **Task 6 a)** – interpolate linearly **within each city**, at most **7** consecutive days, and only **between** two known values.

In [ ]:
air_interp = air.copy()

# TODO group by "City", take the columns `cols` and use transform with a function that calls
#      s.interpolate(limit=7, limit_area="inside")
air_interp[cols] = ...

print("Cells filled:", (air[cols].isna() & air_interp[cols].notna()).sum().sum())

In [ ]:
# ✅ Check
checks.task_6a(air, air_interp, cols)

🧩 **Task 6 b)** – which method is better for PM2.5 in Delhi, and does it depend on the gap length?
We use the masking idea from section 4: hide runs of known days, fill them, measure the error.

In [ ]:
delhi = air.loc[air["City"] == "Delhi"].set_index("Date")["PM2.5"]


def gap_mae(s, gap, method, n_gaps=60, seed=0):
    """Hide n_gaps runs of `gap` consecutive known values, fill each run with `method`, return the mean absolute error."""
    rng = np.random.default_rng(seed)
    values = s.to_numpy()
    starts = [i for i in range(10, len(s) - gap - 10) if not np.isnan(values[i:i + gap]).any()]
    errors = []
    for start in rng.choice(starts, size=n_gaps, replace=False):
        masked = s.copy()
        masked.iloc[start:start + gap] = np.nan
        # TODO 1) fill `masked`: masked.ffill() if method == "ffill", otherwise masked.interpolate(method=method)
        filled = ...
        # TODO 2) mean absolute error between filled and s on the hidden positions start:start + gap
        errors.append(...)
    return np.mean(errors)


gaps = [1, 3, 7, 14]
pd.DataFrame({m: [gap_mae(delhi, g, m) for g in gaps] for m in ["ffill", "linear"]},
             index=pd.Index(gaps, name="gap (days)")).round(1)

In [ ]:
# ✅ Check
checks.task_6b(gap_mae, delhi)

✏️ **Task 6 c)**

1. Which method wins, and how does the error change with the gap length? What does that suggest for the value of `limit`?
2. Mumbai has no AQI for **1222 consecutive days**. What should a Silver table contain for those days, and why?
3. ⭐ Add `"time"` and `"polynomial"` to the comparison (polynomial needs `order=2` – adapt `gap_mae`). Why is `"time"` identical to `"linear"` here?

# 7. ⭐ Final task: Bronze → Silver cleaning as code (≈ 3 min in class + at home)

In Exercise 4 we sketched a lakehouse with **Bronze → Silver → Gold** zones. Here we write the **Bronze → Silver** step for the car table – only the missing-value part
(outliers come next week). In a real platform this is a function that runs on every new load, so it must be:
* **deterministic** – same input, same output; never changes its input,
* **documented** – every decision is written to a **decision log** (what, how many rows, why),
* **verified** – **validation rules** run on the result, and the load stops if a rule fails (data quality *as code*, like tests for software).

Start with steps 2–3 in class and finish at home. It is not graded; the check below prints ✅ or ❌ for each step separately.

In [ ]:
def group_mode(df, by, col):
    """For every row: the most common non-missing value of `col` within its `by` group (NaN if the group has none)."""
    return df.groupby(by)[col].transform(lambda s: s.mode().iloc[0] if s.notna().any() else np.nan)


def clean_cars(df):
    """Bronze -> Silver for the cars table. Returns (clean copy, decision log). Never changes `df`."""
    df = df.copy()
    log = []

    def note(step, column, rows, reason):
        log.append({"step": step, "column": column, "rows changed": int(rows), "reason": reason})

    # 1) Full duplicates (done for you – this is the pattern for every step: change, then note)
    n = len(df)
    df = df.drop_duplicates()
    note(1, "all", n - len(df), "identical rows describe the same car twice (uniqueness)")

    # TODO 2) Transmission Type: the text "UNKNOWN" is a missing value -> NaN
    ...

    # TODO 3) MSRP == 2000 is a placeholder for "price unknown":
    #         add a boolean column MSRP_unknown, then replace 2000 with NaN
    ...

    # TODO 4) Engine Fuel Type missing -> the most common fuel type of the same Make + Model (use group_mode)
    ...

    # TODO 5) Engine Cylinders: electric cars ("Engine Fuel Type" == "electric") with NaN -> 0.
    #         Rotary Mazdas (RX-7, RX-8) stay NaN – not applicable.
    ...

    # TODO 6) Number of Doors missing -> the most common value of the same Make + Model (stays NaN if the group has none)
    ...

    # TODO 7) Engine HP:
    #         - add a boolean column Engine_HP_missing BEFORE filling,
    #         - fill with the median of the same Make + Model + Engine Fuel Type,
    #         - what is still missing: the median of the same Engine Fuel Type
    #           (an electric car should get an electric median, not a petrol one),
    #         - a fuel type with no known HP at all: the median of the same Make + Model
    ...

    # TODO 8) Market Category missing means "no special category" -> the text "None"
    ...

    return df, pd.DataFrame(log)


silver, decision_log = clean_cars(cars)
decision_log

Now the **validation rules**. The same rules run on Bronze (they should fail) and on Silver (they should all pass).
In a real pipeline a failed rule stops the load before bad data reaches the Gold zone.

In [ ]:
def validate(df):
    """Data quality rules for the Silver cars table. Returns the names of the rules that FAIL."""
    rules = {
        "no full duplicates": df.duplicated().sum() == 0,                       # example
        # TODO one boolean expression per rule:
        "no placeholder price (MSRP == 2000)": ...,
        "no text code UNKNOWN in Transmission Type": ...,
        "Engine HP is complete": ...,
        "only Mazda RX-7/RX-8 may lack Engine Cylinders": ...,
    }
    return [name for name, ok in rules.items() if not ok]


print("Bronze fails:", validate(cars))
print("Silver fails:", validate(silver))

In [ ]:
# ✅ Check – every step separately
checks.task_7(cars, silver, decision_log, validate)

✏️ **Decision log review** – for three steps of your choice, write the main **risk** of the decision and how you would detect it later.

| Step | Decision | Risk / weakness | How would you notice it in production? |
|---|---|---|---|
| | | | |
| | | | |
| | | | |

⭐⭐ **More:**
* All 18 Tesla Model S rows got the electric median of 140 HP. Is that plausible? Why did the fallback produce it, and what would be better?
* Add a rule to `validate` that the share of `MSRP_unknown` is not higher than in the last load (+ a tolerance). Why is a *rate* rule useful when every row looks fine?

# Summary

* Missing values hide: **profile** every column (`isna`, `nunique`, the share of the most common value) and look for placeholders and text codes. Separate **unknown** from **not applicable**.
* **Data can reject MCAR** (rates by group, chi-square, t-test, a model predicting the gaps), but it can never prove MAR – MNAR needs domain knowledge or extra data. Read p-values together with rates and sample size.
* The mechanism decides what works: deletion and mean imputation are only safe under MCAR; regression-based methods handle MAR; nothing fixes MNAR from the observed data alone.
* Single imputation **shrinks the spread**; stochastic regression keeps it.
* Compare imputers by **hiding known values** and measuring the error *and* the distribution. Simple, explainable group methods are often almost as good as KNN or MICE. KNN needs **scaled** features and more than one column.
* Learn imputation values **inside a pipeline** on training data only; **never** use the target. Keep a **missing indicator** when missingness carries information.
* Time series: fill **within each series**, with a **limit**; long gaps stay missing.
* In a data platform, cleaning is code: a deterministic function, a **decision log** and **validation rules**.

# Before the homework

The Week 5 assignment asks you to work with a **small** table (10–15 rows). A few things work differently at that size:

* **Tests have almost no power.** With 10–15 rows, a chi-square or t-test will rarely be significant – and a large p-value does **not** show MCAR.
  Argue the mechanism mainly from *how the data was collected*: who or what could have caused each gap? Use rates by group only as a hint.
* **Interpolation needs a natural order** of the rows (time, position along a route…). If your rows have no meaningful order, interpolation is meaningless – choose a dataset where it is, or explain why you use another method.
* **Regression imputation needs a related column** to predict from. Check that the relationship exists in your rows.
* **You don't know the true missing values.** To compare methods anyway, borrow the idea from section 4: hide a value you *do* know, fill it, and see how close each method gets.
  With so few rows, hide each known value **one at a time** instead of a random 20 %.
* **"Compare the results" means more than one number:** also look at what each method does to the mean, the spread and the plausibility of the filled values.

# Sources and further reading

* Datasets: [Kaggle: Titanic](https://www.kaggle.com/c/titanic) · [Kaggle: Air Quality Data in India](https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india) (Vopani) · [Kaggle: Car Features and MSRP](https://www.kaggle.com/datasets/CooperUnion/cardataset) (CooperUnion)
* The diagram in `images/` is adapted from the Kaggle notebook *A Guide to Handling Missing values in Python* (Parul Pandey).
* S. van Buuren: [*Flexible Imputation of Missing Data*](https://stefvanbuuren.name/fimd/), 2nd ed. – free online; chapters 1–3 cover mechanisms and why single imputation understates uncertainty.
* [pandas – Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html) · [scikit-learn – Imputation of missing values](https://scikit-learn.org/stable/modules/impute.html) · [missingno](https://github.com/ResidentMario/missingno)